In [37]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [38]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)



def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [39]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "'json' or 'python' or 'regex'", 
        "solution_criteria": "The characterstics that the solutions should have to fulfil the task"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [40]:
dataset = generate_dataset()


with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [41]:
def run_prompt(test_case):
    #combines boths the prompts and the test cases and provide these as messages to claude
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}

    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output



In [42]:
#pases in the test cases generated by a model, and the ouptuts to those test cases
def grade_by_model(test_case, output):
       # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    Solution Criteria: {test_case["solution_criteria"]}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [43]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [44]:
def run_test_case(test_case):
    #returns json with the test case, and output from claude with the score
    output = run_prompt(test_case)

    #todo -grading
    model_grade = grade_by_model(test_case,output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    #score = 10

    syntax_score = grade_syntax(output, test_case)
    score = (model_score+syntax_score)/2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [45]:
def run_eval(test_case):
    results =  []

    total = 0
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

        total+=result["score"]
    avg = total/len(dataset)
    print(avg)


    average = []
    return results

In [46]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)

8.166666666666666


In [ ]:
print(json.dumps(results, indent = 2))

[
  {
    "output": "\nimport re\n\ndef extract_account_id(arn):\n    match = re.search(r':(\\d{12}):', arn)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Extract the AWS account ID from an ARN (Amazon Resource Name) string",
      "format": "regex",
      "solution_criteria": "The regex should match and capture the 12-digit account ID from ARNs in the format arn:aws:service:region:account-id:resource"
    },
    "score": 8.0,
    "reasoning": "The solution works for well-formed, standard ARNs but is fragile. It would incorrectly extract digits from malformed ARNs (e.g., 'arn:aws:service:1234567890123:account-id:resource' would match the wrong field). The regex lacks context awareness of ARN structure. A more robust solution would validate the full ARN format before extraction, or at minimum verify the digit group is in the correct position (5th colon-separated field). The code lacks error handling documentation and type hints."
  },
  {
    "outpu

: 